## Homework 3: Recommender System
Owen Randolph, DSCI-590: Applied Data Science, Fall 2024

A item-item collaborative-based filtering system will be created recommending the top 5 items to the user based on their choice.  I'll be using the electronics marketing bias dataset found at https://github.com/MengtingWan/marketBias/tree/master/data, cited at the end of this notebook. 

In [556]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [557]:
#Load dataset and check
df = pd.read_csv("df_electronics.csv")
df.head()

,item_id,user_id,rating,timestamp,model_attr,category,brand,year,user_attr,split
0,0,0,5.0,1999-06-13,Female,Portable Audio & Video,NaN,1999,NaN,0
1,0,1,5.0,1999-06-14,Female,Portable Audio & Video,NaN,1999,NaN,0
2,0,2,3.0,1999-06-17,Female,Portable Audio & Video,NaN,1999,NaN,0
3,0,3,1.0,1999-07-01,Female,Portable Audio & Video,NaN,1999,NaN,0
4,0,4,2.0,1999-07-06,Female,Portable Audio & Video,NaN,1999,NaN,0


In [558]:
# Check the number of rows and columns of the dataset
df.shape

(1292954, 10)

In [559]:
# Check the data types in columns
df.dtypes

item_id         int64
user_id         int64
rating        float64
timestamp      object
model_attr     object
category       object
brand          object
year            int64
user_attr      object
split           int64
dtype: object

In [560]:
# Check Missing Values
df.isnull().sum()

item_id             0
user_id             0
rating              0
timestamp           0
model_attr          0
category            0
brand          961834
year                0
user_attr     1118830
split               0
dtype: int64

In [561]:
# Drop Feature variables with high number of missing values and categories that won't be used in recommendation
df1 = df.drop(['brand', 'split', 'model_attr'], axis=1)
df = df.drop(['brand', 'user_attr', 'split', 'timestamp', 'model_attr', 'year'], axis=1)
df.head()

,item_id,user_id,rating,category
0,0,0,5.0,Portable Audio & Video
1,0,1,5.0,Portable Audio & Video
2,0,2,3.0,Portable Audio & Video
3,0,3,1.0,Portable Audio & Video
4,0,4,2.0,Portable Audio & Video


In [562]:
# Filter out users that have given over 5 ratings
user_counts = df.user_id.value_counts()
users_more_than_one = user_counts[user_counts > 5].index
df = df[df.user_id.isin(users_more_than_one)]

In [563]:
df_sorted_desc = df.sort_values(by='user_id', ascending=False)
df_sorted_desc

,item_id,user_id,rating,category
1153371,6726,1034031,3.0,Camera & Photo
1153250,6600,1034031,4.0,Computers & Accessories
1153308,2586,1034031,3.0,Camera & Photo
1153269,2981,1034031,3.0,Camera & Photo
1153709,3404,1034031,4.0,Camera & Photo
...,...,...,...,...
14933,651,28,4.0,Camera & Photo
84919,1776,28,3.0,Camera & Photo
541805,7213,28,3.0,Headphones
79078,1857,28,5.0,Portable Audio & Video


In [564]:
# User Rating Frequency
df.user_id.value_counts()

user_id
142967     41
30661      38
89185      37
80476      34
46878      34
           ..
49146       6
239021      6
238968      6
237388      6
1034031     6
Name: count, Length: 1158, dtype: int64

In [565]:
# Show the number of users
user_unique = df.user_id.nunique()
print(user_unique)

1158


In [566]:
# Show the number of items
item_unique = df.item_id.nunique()
print(item_unique)

2883


In [567]:
# Product categories
print(df.category.unique())

['Portable Audio & Video' 'Camera & Photo' 'Computers & Accessories'
 'Headphones' 'Accessories & Supplies' 'Television & Video'
 'Car Electronics & GPS' 'Home Audio' 'Security & Surveillance'
 'Wearable Technology']


### Item-Item Recommender System
This model will use a collaborative-based filtering algorithm to give a recommendation for an item based on other items that have been rated similarly to it.

In [568]:
# Create sparse user-rating matrix
matrix = df.pivot_table(index = 'item_id', columns = 'user_id', values = 'rating')
matrix.fillna(0, inplace = True)
matrix

user_id,28,158,269,789,822,1007,1156,1170,1188,1277,...,893097,899072,899955,907563,911696,919192,919932,926621,978504,1034031
item_id,,,,,,,,,,,,,,,,,,,,,
0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14,0.0,4.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9504,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9509,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9510,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [569]:
# Standardize the pivot table
scaler = StandardScaler(with_mean=True, with_std=True)
matrix_normalized = scaler.fit_transform(matrix)

In [570]:
# Similarity matrix for all items
similarity_score = cosine_similarity(matrix_normalized)

We create function called item_item_recommend() which recommends the top 3 items to the user based on their choice.
1. The code finds the numerical index of the chosen item in the matrix
2. It sorts the similarity scores for that item in descending order
3. It selects the top 3 most similar items
4. It retrieves the category of the recommended items
5. It returns the information as a list

In [571]:
def recommend(item_id):
    # Returns the numerical index for the item_id
    index = np.where(matrix.index == item_id)[0][0]

    # Sorts the similarities for the item_id in descending order
    similar_items = sorted(list(enumerate(similarity_score[index])), key=lambda x: x[1], reverse=True)[1:6]

    # To return results in list format
    data = []
    seen_items = set()  # Track seen item_ids

    for idx, similarity in similar_items:
        # Get the item details by index
        temp_df = df[df['item_id'] == matrix.index[idx]]

        # Ensure the item_id is not added more than once
        unique_item_ids = temp_df['item_id'].unique()
        for unique_item_id in unique_item_ids:
            if unique_item_id in seen_items:
                continue  # Skip if already added

            # Add item_id and the first category
            item = [unique_item_id]
            category = temp_df[temp_df['item_id'] == unique_item_id]['category'].iloc[0]
            item.append(category)

            data.append(item)
            seen_items.add(unique_item_id)  # Mark this item_id as seen

    return data

In [572]:
# Example of item-item recommender function in use
recommend(9537)

[[119, 'Computers & Accessories'],
 [151, 'Computers & Accessories'],
 [4286, 'Headphones'],
 [3, 'Camera & Photo'],
 [4694, 'Computers & Accessories']]

Citation:
https://www.geeksforgeeks.org/build-a-recommendation-engine-with-collaborative-filtering/

### Recommendation based on Deep Learning
This will be a different approach using deep learning techniques using Neural Collaborative Filtering (NCF).  A version of the dataset that has more features will be used in order to capture more diversity on user-item interaction and preferences.

In [580]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense

In [582]:
df.head()

,item_id,user_id,rating,category
28,0,0,2.0,Portable Audio & Video
158,2,1,2.0,Camera & Photo
183,4,1,4.0,Camera & Photo
271,4,2,5.0,Camera & Photo
279,5,2,5.0,Camera & Photo


In [583]:
df.shape

(9509, 4)

In [584]:
# Encode user_id and item_id as integers
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['item_id'] = item_encoder.fit_transform(df['item_id'])

In [585]:
# Train Test Split
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [586]:
# Input layers
user_input = Input(shape=(1,), name='user_input')
item_input = Input(shape=(1,), name='item_input')

# Embedding layers
user_embedding = Embedding(input_dim=user_unique, output_dim=50, name='user_embedding')(user_input)
item_embedding = Embedding(input_dim=item_unique, output_dim=50, name='item_embedding')(item_input)

# Flatten embeddings
user_vec = Flatten()(user_embedding)
item_vec = Flatten()(item_embedding)

# Concatenate user and item vectors
concat = Concatenate()([user_vec, item_vec])

# Dense layers
dense = Dense(128, activation='relu')(concat)
dense = Dense(64, activation='relu')(dense)
output = Dense(1, activation='linear')(dense)

# Build and compile model
model = Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer='adam', loss='mean_squared_error')

#### Train the Model

In [587]:
history = model.fit(
    [train['user_id'], train['item_id']],  # Input user and item IDs
    train['rating'],                       # Target ratings
    epochs=10,                             # Number of epochs (adjust as needed)
    batch_size=32,                         # Batch size (adjust as needed)
    validation_split=0.1,                  # Use 10% of training data for validation
    verbose=1
)

Epoch 1/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 8.4232 - val_loss: 0.9840
Epoch 2/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 841us/step - loss: 0.7738 - val_loss: 1.0144
Epoch 3/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 865us/step - loss: 0.5764 - val_loss: 1.0853
Epoch 4/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 831us/step - loss: 0.4894 - val_loss: 1.0864
Epoch 5/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 800us/step - loss: 0.4031 - val_loss: 1.0889
Epoch 6/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 851us/step - loss: 0.3095 - val_loss: 1.1380
Epoch 7/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 825us/step - loss: 0.2684 - val_loss: 1.2867
Epoch 8/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1772 - val_loss: 1.2071
Epoch 9/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 811us/step - loss: 0.1390 - val_loss: 1.2674
Epoch 10/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 811us/step - loss: 0.1062 - val_loss: 1.3002


In [588]:
# Evaluate the model on the test data
test_loss = model.evaluate([test['user_id'], test['item_id']], test['rating'])
print(f"Test Loss: {test_loss}")

60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2444
Test Loss: 1.304826259613037


In [593]:
# Example of a Prediction
# Convert inputs to numpy arrays before transforming
user_id = np.array([822])  # Convert to NumPy array
item_id = np.array([98])   # Convert to NumPy array

# Perform prediction
predicted_rating = model.predict([user_encoder.transform(user_id), item_encoder.transform(item_id)])
print(f"Predicted Rating: {predicted_rating}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Predicted Rating: [[4.002705]]


This model is not hyper-parameter tuned.  Although training loss decreased continuously throughout the training, validation loss sees fluctuation, indicating possible overfitting of the model.  We could introduce L2 regularization or drop out to see if that will prevent this overfitting.  

In [ ]:
Citations:

GeeksforGeeks. "Build a Recommendation Engine with Collaborative Filtering." GeeksforGeeks, GeeksforGeeks, https://www.geeksforgeeks.org/build-a-recommendation-engine-with-collaborative-filtering/

Chen, Yihong. Neural Collaborative Filtering. GitHub, https://github.com/yihong-chen/neural-collaborative-filtering.